In [1]:
import xarray as xr
import sys, os
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# Local Imports
curdir = os.getcwd()
#print(curdir)
sys.path.insert(0, curdir+"/../data_processing")
from ceres_ebaf import *

sys.path.insert(0, curdir+"/../visualization")
from ceres_ebaf_plotting import *

plt.rcParams['figure.figsize'] = [10, 3.5]

In [9]:
file_name = "/Users/mawa7160/dev/data/CERES/EBAF/CERES_EBAF-TOA_Full_2022_01.nc"
ceres_ds = xr.open_dataset(file_name)
ceres_full_years = ceres_ds
solar_full_data = ceres_full_years["solar_mon"]

def show_and_tell(data_set, title="NH-SH-Global"):
    plot_hemisphere_and_global_by_year(data_set.year, data_set, fixed_ylim=False, title=title)
    nh_error = mean_squared_error(data_set["nh"], data_set["global"])
    global_avg = np.average(data_set["global"])
    nh_avg = np.average(data_set["nh"])
    sh_avg = np.average(data_set["sh"])
    print("Mean Squared error from NH to Global: "+str(nh_error))
    print("NH Total Avg: "+str(nh_avg)+" W m^-2")
    print("Global Total Avg: "+str(global_avg)+" W m^-2")
    print("NH-SH Difference Difference: "+str(nh_avg - sh_avg)+" W m^-2")
    print("NH-Glob Difference Difference: "+str(nh_avg - global_avg)+" W m^-2")
    return global_avg, nh_avg-sh_avg

In [10]:
bad_time_avg = create_hemisphere_data(solar_full_data, start_yr="2001", start_mon="01", 
                end_yr="2022", end_mon="01", time_weighting=-1, space_weighting=1)
print("Averaging without days of the month")
bad_time_glob, bad_time_diff = show_and_tell(bad_time_avg)

In [11]:
ok_time_avg = create_hemisphere_data(solar_full_data, start_yr="2001", start_mon="01", 
                end_yr="2022", end_mon="01", time_weighting=0, space_weighting=1)
print("Averaging with days of the month")
ok_time_glob, ok_time_diff = show_and_tell(ok_time_avg)

In [12]:
best_time_avg = create_hemisphere_data(solar_full_data, start_yr="2001", start_mon="01", 
                end_yr="2022", end_mon="01", time_weighting=1, space_weighting=1)
print("Averaging Empirical Best")
best_time_glob, best_time_diff = show_and_tell(best_time_avg,  title="NH-SH-Global With Numerical February Fix")

In [13]:
good_time_avg = create_hemisphere_data(solar_full_data, start_yr="2001", start_mon="01", 
                end_yr="2022", end_mon="01", time_weighting=2, space_weighting=1)
print("Averaging with Jake fix")
good_time_glob, good_time_diff = show_and_tell(good_time_avg, title="NH-SH-Global With First and Last Month Fix")

In [20]:
bad = [np.average(bad_time_avg["global"]), np.abs(np.average(bad_time_avg["nh"])-np.average(bad_time_avg["global"]))]
ok = [np.average(ok_time_avg["global"]), np.abs(np.average(ok_time_avg["nh"])-np.average(ok_time_avg["global"]))]
good = [np.average(good_time_avg["global"]), np.abs(np.average(good_time_avg["nh"])-np.average(good_time_avg["global"]))]
best = [np.average(best_time_avg["global"]), np.abs(np.average(best_time_avg["nh"])-np.average(best_time_avg["global"]))]

## Spatial Averaging Comparison

In [21]:
cos_spatial_weighting = create_hemisphere_data(solar_full_data, start_yr="2001", start_mon="01",
                end_yr="2022", end_mon="01", time_weighting=1, space_weighting=2)
print("Spatial Averaging as cosine")
good_spat_glob, good_spat_diff = show_and_tell(cos_spatial_weighting, title="NH-SH-Global With First and Last Month Fix")

In [22]:
geo_spatial_weighting = create_hemisphere_data(solar_full_data, start_yr="2001", start_mon="01",
                end_yr="2022", end_mon="01", time_weighting=1, space_weighting=1)
print("Spatial Averaging as Geodetic weighting")
great_spat_glob, great_spat_diff = show_and_tell(geo_spatial_weighting, title="NH-SH-Global With First and Last Month Fix")

In [23]:
bad = create_hemisphere_data(solar_full_data, start_yr="2001", start_mon="01",
                end_yr="2022", end_mon="01", time_weighting=1, space_weighting=-1)
print("Spatial Averaging as no latitude weighting")
bad_spat_glob, bad_spat_diff = show_and_tell(bad, title="NH-SH-Global With First and Last Month Fix")

In [25]:
plt.plot([bad_spat_glob, good_spat_glob, great_spat_glob, bad_time_glob, ok_time_glob, good_spat_glob, best_time_glob], 'x')


In [33]:
plt.plot([bad_spat_diff, good_spat_diff, great_spat_diff, bad_time_diff, ok_time_diff, good_spat_diff, best_time_diff], 'x')
print(min(np.abs([bad_spat_diff, good_spat_diff, great_spat_diff, bad_time_diff, ok_time_diff, good_spat_diff, best_time_diff])))